In [1]:
import findspark
import pyspark

findspark.init()

In [2]:
sc = pyspark.SparkContext.getOrCreate()

In [3]:
inputRDD = sc.textFile("ReviewsSample.csv")
inputRDD.collect()

['Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text',
 '1,B1,A2,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.',
 '2,B1,A4,dll pa,0,0,4,1346976000,Not as Advertised,"Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor intended to represent the product as ""Jumbo""."',
 '3,B1,A5,"Natalia Corres ""Natalia Corres""",1,1,1,1219017600,"""Delight"" says it all","This is a confection that has been around a few centuries.  It is a light, pillowy citrus gelatin with nuts - in this case Filberts. And it is cut into tiny squares and then liberally coated with powdered sugar.  And it is a tiny m

In [4]:
filteredRDD = inputRDD.filter(lambda line: line.find('Id') == -1)
filteredRDD.collect()

['1,B1,A2,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.',
 '2,B1,A4,dll pa,0,0,4,1346976000,Not as Advertised,"Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor intended to represent the product as ""Jumbo""."',
 '3,B1,A5,"Natalia Corres ""Natalia Corres""",1,1,1,1219017600,"""Delight"" says it all","This is a confection that has been around a few centuries.  It is a light, pillowy citrus gelatin with nuts - in this case Filberts. And it is cut into tiny squares and then liberally coated with powdered sugar.  And it is a tiny mouthful of heaven.  Not too chewy, and very flavorful.  I highly recommend this yummy treat.  If you are

In [5]:
mappedRDD = filteredRDD.map(lambda line: (line.split(",")[2], line.split(",")[1]))
mappedRDD.collect()

[('A2', 'B1'),
 ('A4', 'B1'),
 ('A5', 'B1'),
 ('A1', 'B2'),
 ('A2', 'B3'),
 ('A3', 'B3'),
 ('A4', 'B3'),
 ('A5', 'B3'),
 ('A4', 'B4'),
 ('A2', 'B5'),
 ('A4', 'B5'),
 ('A2', 'B1'),
 ('A4', 'B5'),
 ('A5', 'B5')]

In [6]:
mappedRDD = mappedRDD.distinct()
mappedRDD.collect()

[('A2', 'B1'),
 ('A2', 'B3'),
 ('A3', 'B3'),
 ('A4', 'B4'),
 ('A4', 'B5'),
 ('A5', 'B5'),
 ('A4', 'B1'),
 ('A5', 'B1'),
 ('A1', 'B2'),
 ('A4', 'B3'),
 ('A5', 'B3'),
 ('A2', 'B5')]

In [7]:
reducedRDD = mappedRDD.reduceByKey(lambda x, y: x + "," + y)
reducedRDD.collect()

[('A4', 'B4,B5,B1,B3'),
 ('A5', 'B5,B1,B3'),
 ('A2', 'B1,B3,B5'),
 ('A3', 'B3'),
 ('A1', 'B2')]

In [8]:
mappedRDD = reducedRDD.map(lambda x: (x[0], x[1].split(",")))
mappedRDD.collect()

[('A4', ['B4', 'B5', 'B1', 'B3']),
 ('A5', ['B5', 'B1', 'B3']),
 ('A2', ['B1', 'B3', 'B5']),
 ('A3', ['B3']),
 ('A1', ['B2'])]

In [9]:
def count_pairs(line):
    line.sort()
    for i in range(len(line) - 1):
        for j in range(i + 1, len(line)):
            yield line[i] + "," + line[j]

In [10]:
mappedRDD = mappedRDD.flatMapValues(count_pairs)
mappedRDD.collect()

[('A4', 'B1,B3'),
 ('A4', 'B1,B4'),
 ('A4', 'B1,B5'),
 ('A4', 'B3,B4'),
 ('A4', 'B3,B5'),
 ('A4', 'B4,B5'),
 ('A5', 'B1,B3'),
 ('A5', 'B1,B5'),
 ('A5', 'B3,B5'),
 ('A2', 'B1,B3'),
 ('A2', 'B1,B5'),
 ('A2', 'B3,B5')]

In [11]:
mappedRDD = mappedRDD.map(lambda x: (x[1], x[0]))
mappedRDD.collect()

[('B1,B3', 'A4'),
 ('B1,B4', 'A4'),
 ('B1,B5', 'A4'),
 ('B3,B4', 'A4'),
 ('B3,B5', 'A4'),
 ('B4,B5', 'A4'),
 ('B1,B3', 'A5'),
 ('B1,B5', 'A5'),
 ('B3,B5', 'A5'),
 ('B1,B3', 'A2'),
 ('B1,B5', 'A2'),
 ('B3,B5', 'A2')]

In [12]:
countRDD = mappedRDD.countByKey()
for key, value in countRDD.items():
    print(key, value)

B1,B3 3
B1,B4 1
B1,B5 3
B3,B4 1
B3,B5 3
B4,B5 1


In [13]:
countedRDD = sc.parallelize(countRDD.items())
countedRDD.collect()

[('B1,B3', 3),
 ('B1,B4', 1),
 ('B1,B5', 3),
 ('B3,B4', 1),
 ('B3,B5', 3),
 ('B4,B5', 1)]

In [14]:
orderedByValueRDD = countedRDD.sortBy(lambda x: x[1], ascending=False)
orderedByValueRDD.collect()

[('B1,B3', 3),
 ('B1,B5', 3),
 ('B3,B5', 3),
 ('B1,B4', 1),
 ('B3,B4', 1),
 ('B4,B5', 1)]

In [15]:
orderedByValueRDD.take(10)

[('B1,B3', 3),
 ('B1,B5', 3),
 ('B3,B5', 3),
 ('B1,B4', 1),
 ('B3,B4', 1),
 ('B4,B5', 1)]

In [17]:
print(orderedByValueRDD.getNumPartitions())

16


In [18]:
orderedByValueRDD = orderedByValueRDD.coalesce(2)

In [19]:
orderedByValueRDD.saveAsTextFile("output")